# Multi-Model Comprehensive Benchmark Evaluator 
### Evaluasi Komparatif 14 Model Lengkap Lintas Spektrum: SLM Lokal, Open-Source 7B-30B, & Proprietary Cloud (Claude Opus 5, GPT-5.6, Gemini 2.5)

Notebook ini mengeksekusi **Evaluasi Komparatif Multi-Model Komprehensif** secara otomatis dan objektif pada **101 butir ujian held-out test set TelAgentBench Indonesia** (55 Single-Turn + 46 Multi-Turn) & 23 Skema API BSS TelkomNusa.

### 5 Pilar Kelompok Model yang Diuji:
1. **Model Terlatih Kita**: `TelkomNusa-SLM-3B` & `TelkomNusa-LLM-7B`
2. **Open SLM (GPU Lokal <= 4B)**: `Qwen2.5-3B-Base-Zero`, `Qwen2.5-3B-Base-CoT`, `Llama-3.2-3B`, `Phi-4-mini-3.8B`
3. **Open-Source MoE & 7B-30B (OpenRouter)**: `Gemma-4-26B-A4B-MoE`, `GLM-5.2-Reasoning`, `MiniMax-M3-Agentic`, `Nemotron-3.5-Lightning`
4. **Proprietary Cloud Models (Claude Opus 5, GPT-5.6-Terra, GPT-4o)**: `Claude-Opus-5`, `GPT-5.6-Terra`, `GPT-4o`
5. **Google Cloud**: `Gemini-2.5-Flash`


## Section 0: Setup Dependencies & Konfigurasi API Multi-Platform


In [ ]:
# 1. Pasang dependensi lokal, library API cloud, serta modul NLP (BLEU & ROUGE)
!pip install -q -U peft accelerate bitsandbytes scikit-learn pandas matplotlib tabulate tqdm huggingface_hub openai google-generativeai requests nltk rouge-score

import os, sys, re, ast, json, time, gc, torch, requests, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import List, Dict, Any, Tuple, Optional
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from openai import OpenAI

# PATCH KOMPATIBILITAS TRANSFORMERS (ANTI SLIDINGWINDOWCACHE ERROR)
try:
    import transformers.cache_utils as _tcu
    if not hasattr(_tcu, "SlidingWindowCache"):
        _tcu.SlidingWindowCache = getattr(_tcu, "DynamicCache", object)
        print("[OK] Patch SlidingWindowCache aktif.")
except Exception as e:
    pass

try:
    import transformers.utils.import_utils as _tfu
    _tfu.is_torchvision_available = lambda: False
except Exception:
    pass

# Modul BLEU & ROUGE
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

smoothie = SmoothingFunction().method1
rouge_evaluator = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

# KONFIGURASI API KEYS MULTI-PLATFORM
JUSTWOKER_API_KEY  = os.environ.get("JUSTWOKER_API_KEY", "YOUR_API_KEY_HERE")
BLUESMINDS_API_KEY = os.environ.get("BLUESMINDS_API_KEY", "YOUR_API_KEY_HERE")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY_HERE")
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")

# 1. Inisialisasi Klien JustWoker (Claude Opus 5)
client_justwoker = None
if JUSTWOKER_API_KEY:
    try:
        client_justwoker = OpenAI(
            api_key=JUSTWOKER_API_KEY,
            base_url="https://api.justwoker.icu/v1"
        )
        print("[OK] JustWoker Client Siap (Model: claude-opus-5)!")
    except Exception as e:
        print(f"[WARNING] JustWoker init gagal: {e}")

# 2. Inisialisasi Klien Bluesminds (OpenAI Compatible)
client_bluesminds = None
if BLUESMINDS_API_KEY:
    try:
        client_bluesminds = OpenAI(
            api_key=BLUESMINDS_API_KEY,
            base_url="https://api.bluesminds.com/v1"
        )
        print("[OK] Bluesminds Client Siap (Model: gpt-5.6-terra, gpt-4o)!")
    except Exception as e:
        print(f"[WARNING] Bluesminds init gagal: {e}")

# 3. Inisialisasi Klien OpenRouter
client_openrouter = None
if OPENROUTER_API_KEY:
    try:
        client_openrouter = OpenAI(
            api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1"
        )
        print("[OK] OpenRouter Client Siap (Model: Gemma 4, GLM 5.2, MiniMax M3, Nemotron 3.5)!")
    except Exception as e:
        print(f"[WARNING] OpenRouter init gagal: {e}")

# 4. Inisialisasi Klien Google Gemini
if GEMINI_API_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        print("[OK] Google Gemini Client Siap (Model: Gemini 2.5 Flash)!")
    except Exception as e:
        print(f"[WARNING] Gemini init gagal: {e}")

if torch.cuda.is_available():
    print(f"[OK] GPU Lokal Terdeteksi: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")
else:
    print("[INFO] GPU Lokal tidak aktif (Mode Cloud API / CPU)")


## Section 1: Pemuatan Dataset Held-Out Benchmark TelAgentBench Clean


In [ ]:
POSSIBLE_BENCH_PATHS = [
    "/kaggle/input/datasets/mrislantristansyah/id-telagentbench/dataset_telagentbench_indonesia_clean",
    "/kaggle/input/id-telagentbench/dataset_telagentbench_indonesia_clean",
    "/kaggle/input/id-telagentbench",
    "/kaggle/input/dataset-telagentbench-indonesia-clean",
    "/kaggle/input/dataset_telagentbench_indonesia_clean",
    "/content/dataset_telagentbench_indonesia_clean",
    "/content/dataset_telagentbench_indonesia_clean/dataset_telagentbench_indonesia_clean",
    "/content/id-telagentbench/dataset_telagentbench_indonesia_clean",
    "dataset_telagentbench_indonesia_clean",
    "../dataset_telagentbench_indonesia_clean"
]

DATA_DIR = None
for p in POSSIBLE_BENCH_PATHS:
    if os.path.exists(p):
        if os.path.exists(os.path.join(p, "TelAgent_Action")):
            DATA_DIR = p
            break
        elif os.path.exists(os.path.join(p, "dataset_telagentbench_indonesia_clean", "TelAgent_Action")):
            DATA_DIR = os.path.join(p, "dataset_telagentbench_indonesia_clean")
            break

if DATA_DIR is None:
    search_roots = ["/kaggle/input", "/content", "."]
    for s_root in search_roots:
        if os.path.exists(s_root):
            for root, dirs, files in os.walk(s_root):
                if "TelAgent_Action" in dirs:
                    DATA_DIR = root
                    break
            if DATA_DIR:
                break

if DATA_DIR is None:
    raise FileNotFoundError("Folder benchmark 'TelAgent_Action' tidak ditemukan!")

print(f"[OK] Folder Benchmark Terdeteksi di: {DATA_DIR}")

ACTION_DIR = os.path.join(DATA_DIR, "TelAgent_Action")
SIMPLE_FILE = os.path.join(ACTION_DIR, "telco_simple.json")
SIMPLE_ANS_FILE = os.path.join(ACTION_DIR, "possible_answer", "telco_simple.json")
MULTI_FILE = os.path.join(ACTION_DIR, "telco_multi_turn_stateless_base.json")
MULTI_ANS_FILE = os.path.join(ACTION_DIR, "possible_answer", "telco_multi_turn_stateless_base.json")
TOOLS_FILE = os.path.join(ACTION_DIR, "250918_telcoFC_tools.json")

TELCO_TOOLS = []
with open(TOOLS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line_s = line.strip()
        if line_s:
            try:
                TELCO_TOOLS.append(json.loads(line_s))
            except Exception:
                pass
if not TELCO_TOOLS:
    with open(TOOLS_FILE, "r", encoding="utf-8") as f:
        TELCO_TOOLS = json.load(f)

FORMATTED_TOOLS = []
for t in TELCO_TOOLS:
    if "type" in t and t["type"] == "function":
        FORMATTED_TOOLS.append(t)
    else:
        FORMATTED_TOOLS.append({"type": "function", "function": t})

ALL_TOOL_NAMES = [t.get("function", t).get("name", "") for t in FORMATTED_TOOLS]

simple_instances = []
with open(SIMPLE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            simple_instances.append(json.loads(line))

simple_answers = {}
with open(SIMPLE_ANS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            simple_answers[item["id"]] = item["ground_truth"]

multi_instances = []
with open(MULTI_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            multi_instances.append(json.loads(line))

multi_answers = {}
if os.path.exists(MULTI_ANS_FILE):
    with open(MULTI_ANS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                multi_answers[item["id"]] = item["ground_truth"]

print(f"\n[OK] Dataset Benchmark TelAgentBench Indonesia Clean Berhasil Dimuat:")
print(f"- 23 Skema Tool API BSS   : {len(FORMATTED_TOOLS)} tools")
print(f"- Single-Turn Test Set    : {len(simple_instances)} butir soal")
print(f"- Multi-Turn Test Set     : {len(multi_instances)} butir percakapan")
print(f"- Total Butir Uji Evaluasi: {len(simple_instances) + len(multi_instances)} sampel")


## Section 2: Master Model Registry (Lengkap 14 Model Tanpa Terkecuali)


In [ ]:
# MASTER REGISTRY MODEL BENCHMARK MULTI-PLATFORM (14 MODEL LENGKAP)

MODEL_REGISTRY = {
    #  Group 1: Fine-Tuned TelkomNusa Models
    "TelkomNusa-SLM-3B": {
        "platform": "local_lora",
        "base_model": "Qwen/Qwen2.5-3B-Instruct",
        "adapter": "Rislantrs/telecom-troubleshooter-qwen2.5-3b-lora",
        "params": "3B",
        "category": "Fine-Tuned SLM",
        "mode": "Zero-Shot SFT",
        "active": True
    },
    "TelkomNusa-LLM-7B": {
        "platform": "local_lora",
        "base_model": "Qwen/Qwen2.5-7B-Instruct",
        "adapter": "Rislantrs/telecom-troubleshooter-qwen2.5-7b-lora",
        "params": "7B",
        "category": "Fine-Tuned LLM",
        "mode": "Zero-Shot SFT",
        "active": True
    },

    #  Group 2: Local Open-Weight Baselines (GPU KITA <= 4B)
    "Qwen2.5-3B-Base-Zero": {
        "platform": "local_hf",
        "model_id": "Qwen/Qwen2.5-3B-Instruct",
        "params": "3B",
        "category": "Open SLM",
        "mode": "Zero-Shot",
        "active": True
    },
    "Qwen2.5-3B-Base-CoT": {
        "platform": "local_hf",
        "model_id": "Qwen/Qwen2.5-3B-Instruct",
        "params": "3B",
        "category": "Open SLM",
        "mode": "3-Shot Prompting",
        "active": True
    },
    "Llama-3.2-3B-Instruct": {
        "platform": "local_hf",
        "model_id": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "params": "3B",
        "category": "Open SLM",
        "mode": "Zero-Shot",
        "active": True
    },
    "Phi-4-mini-3.8B": {
        "platform": "local_hf",
        "model_id": "microsoft/Phi-4-mini-instruct",
        "params": "3.8B",
        "category": "Open SLM",
        "mode": "Zero-Shot",
        "active": True
    },

    #  Group 3: Mid-to-Large Open-Weight Models (OPENROUTER CLOUD)
    "Gemma-4-26B-A4B-MoE": {
        "platform": "api_openrouter",
        "model_id": "google/gemma-4-26b-a4b-it:free",
        "params": "26B (3.8B Active)",
        "category": "Open MoE",
        "mode": "Zero-Shot Function Call",
        "active": True
    },
    "GLM-5.2-Reasoning": {
        "platform": "api_openrouter",
        "model_id": "z-ai/glm-5.2:free",
        "params": "18.6B Active",
        "category": "Open Reasoning",
        "mode": "Zero-Shot Reasoning",
        "active": True
    },
    "MiniMax-M3-Agentic": {
        "platform": "api_openrouter",
        "model_id": "minimax/minimax-m3:free",
        "params": "MoE Agentic",
        "category": "Open Agentic",
        "mode": "Zero-Shot Tool Call",
        "active": True
    },
    "Nemotron-3.5-Lightning": {
        "platform": "api_openrouter",
        "model_id": "nvidia/nemotron-3.5-lightning:free",
        "params": "30B-A3B",
        "category": "Open MoE",
        "mode": "Zero-Shot Tool Call",
        "active": True
    },

    #  Group 4: PROPRIETARY CLOUD MODELS (CLAUDE, GPT, GEMINI)
    "Claude-Opus-5": {
        "platform": "api_justwoker",
        "model_id": "claude-opus-5",
        "params": "Proprietary Cloud",
        "category": "Proprietary Cloud",
        "mode": "Zero-Shot Tool Call",
        "active": True
    },
    "GPT-5.6-Terra": {
        "platform": "api_bluesminds",
        "model_id": "gpt-5.6-terra",
        "params": "Proprietary Cloud",
        "category": "Proprietary Cloud",
        "mode": "Zero-Shot Tool Call",
        "active": True
    },
    "GPT-4o": {
        "platform": "api_bluesminds",
        "model_id": "gpt-4o",
        "params": "Proprietary Cloud",
        "category": "Proprietary Cloud",
        "mode": "Zero-Shot Tool Call",
        "active": True
    },
    "Gemini-2.5-Flash": {
        "platform": "api_gemini",
        "model_id": "gemini-2.5-flash",
        "params": "Cloud API",
        "category": "Cloud API",
        "mode": "Zero-Shot Function Call",
        "active": True
    }
}

active_models = [k for k, v in MODEL_REGISTRY.items() if v.get("active", False)]
print(f"[OK] Master Registry Siap! Total {len(active_models)} Model Aktif Siap Diuji:")
for idx, m_name in enumerate(active_models, 1):
    m_info = MODEL_REGISTRY[m_name]
    print(f"{idx:2d}. [{m_info['category']:18s}] {m_name:25s} ({m_info['params']:15s} via {m_info['platform']})")


## Section 3: Universal Parser, Normalizer, Evaluator 5 Pilar + BLEU & ROUGE


In [ ]:
def extract_tool_calls(response_text: str) -> List[Dict[str, Any]]:
    calls = []
    if not response_text:
        return calls
        
    xml_pattern = r"<tool_call>(.*?)</tool_call>"
    xml_matches = re.findall(xml_pattern, response_text, re.DOTALL)
    if xml_matches:
        for match in xml_matches:
            try:
                parsed = json.loads(match.strip())
                if isinstance(parsed, dict) and "name" in parsed:
                    calls.append(parsed)
            except Exception:
                pass
        if calls:
            return calls

    json_block_pattern = r"```(?:json)?\s*([\{\[].*?[\}\]])\s*```"
    json_matches = re.findall(json_block_pattern, response_text, re.DOTALL)
    for jm in json_matches:
        try:
            parsed = json.loads(jm.strip())
            if isinstance(parsed, dict) and "name" in parsed:
                calls.append(parsed)
            elif isinstance(parsed, list):
                for item in parsed:
                    if isinstance(item, dict) and "name" in item:
                        calls.append(item)
        except Exception:
            pass
    if calls:
        return calls

    try:
        raw_p = json.loads(response_text.strip())
        if isinstance(raw_p, dict) and "name" in raw_p:
            return [raw_p]
    except Exception:
        pass

    fn_pattern = r"([A-Za-z0-9_]+__\w+)\((.*?)\)"
    fn_matches = re.findall(fn_pattern, response_text)
    for fname, args_str in fn_matches:
        arg_dict = {}
        kv_pairs = re.findall(r"([a-zA-Z0-9_]+)\s*=\s*(['\"]?)(.*?)\2(?:,|$)", args_str)
        for k, _, v in kv_pairs:
            try:
                arg_dict[k] = ast.literal_eval(v)
            except Exception:
                arg_dict[k] = v
        calls.append({"name": fname, "arguments": arg_dict})

    return calls

def normalize_ground_truth(gt_raw: Any) -> List[Dict[str, Any]]:
    calls = []
    if not gt_raw:
        return calls
    
    items_to_process = []
    if isinstance(gt_raw, list):
        for item in gt_raw:
            if isinstance(item, list):
                items_to_process.extend(item)
            else:
                items_to_process.append(item)
    elif isinstance(gt_raw, dict):
        items_to_process.append(gt_raw)
        
    for item in items_to_process:
        if isinstance(item, dict):
            if "name" in item:
                calls.append({"name": item["name"], "arguments": item.get("arguments", {})})
            else:
                for fname, fargs in item.items():
                    calls.append({"name": fname, "arguments": fargs})
    return calls

def compute_nlg_metrics(candidate_text: str, reference_text: str) -> Tuple[float, float]:
    if not candidate_text or not reference_text:
        return 0.0, 0.0
        
    cand_tokens = candidate_text.strip().lower().split()
    ref_tokens = [reference_text.strip().lower().split()]
    
    try:
        b4 = sentence_bleu(ref_tokens, cand_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie) * 100.0
    except Exception:
        b4 = 0.0
        
    try:
        r_scores = rouge_evaluator.score(reference_text, candidate_text)
        r_l = r_scores['rougeL'].fmeasure * 100.0
    except Exception:
        r_l = 0.0
        
    return round(b4, 2), round(r_l, 2)

def evaluate_instance(
    pred_calls: List[Dict[str, Any]],
    gold_calls: List[Dict[str, Any]]
) -> Tuple[bool, bool, bool, bool, str]:
    if not pred_calls and not gold_calls:
        return True, True, True, True, "Correct (No Tool Needed)"
    
    if not pred_calls and gold_calls:
        return False, False, False, False, "Format / Parsing Error (No Tool Emitted)"
    
    if pred_calls and not gold_calls:
        return False, False, False, True, "Spurious Tool Call (Hallucinated Tool)"

    pred_tool = pred_calls[0].get("name", "").strip().lower()
    gold_tool = gold_calls[0].get("name", "").strip().lower()
    
    tool_correct = (pred_tool == gold_tool)
    if not tool_correct:
        return False, False, False, True, f"Wrong Tool (Pred: {pred_calls[0].get('name', '')} vs Gold: {gold_calls[0].get('name', '')})"

    pred_args = pred_calls[0].get("arguments", {})
    if isinstance(pred_args, str):
        try:
            pred_args = json.loads(pred_args)
        except Exception:
            pred_args = {}
            
    gold_args = gold_calls[0].get("arguments", {})
    if isinstance(gold_args, str):
        try:
            gold_args = json.loads(gold_args)
        except Exception:
            gold_args = {}

    for k, g_val in gold_args.items():
        if k not in pred_args:
            if isinstance(g_val, list) and (None in g_val or "" in g_val):
                continue
            return True, False, False, True, f"Missing Parameter: {k}"
        
        p_val_str = str(pred_args[k]).strip().lower()
        if isinstance(g_val, list):
            allowed_vals = [str(x).strip().lower() for x in g_val if x is not None]
            if allowed_vals and p_val_str not in allowed_vals:
                return True, False, False, True, f"Wrong Value for {k}: Pred='{pred_args[k]}' vs Gold={g_val}"
        else:
            if p_val_str != str(g_val).strip().lower():
                return True, False, False, True, f"Wrong Value for {k}: Pred='{pred_args[k]}' vs Gold='{g_val}'"

    return True, True, True, True, "Correct"

print("[OK] Universal Parser, AST Evaluator, & Modul BLEU/ROUGE Siap!")


## Section 4: Multi-Platform Inference & Evaluation Runner (14 Model Otomatis)


In [ ]:
SYSTEM_PROMPT = "Anda adalah asisten AI resmi Customer Care TelkomNusa. Panggil tool yang sesuai jika diperlukan."
BASE_OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"

FEW_SHOT_COT_PREFIX = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Tolong cek sisa pulsa nomor 08123456789"},
    {"role": "assistant", "content": "Mari kita cek pulsa pelanggan dengan tool bss__check_balance_and_status.\n<tool_call>{\"name\": \"bss__check_balance_and_status\", \"arguments\": {\"msisdn\": \"08123456789\"}}</tool_call>"}
]

all_models_summary = []

for model_name, cfg in MODEL_REGISTRY.items():
    if not cfg.get("active", False):
        continue
        
    print("="*85)
    print(f"[*] MEMULAI EVALUASI MODEL: {model_name} [{cfg['category']}] via {cfg['platform']} ({cfg['mode']})")
    print("="*85)
    
    platform = cfg["platform"]
    model_records = []
    eval_start_time = time.time()
    total_tokens = 0
    
    tokenizer = None
    model = None
    
    # ---- 1. INISIALISASI MODEL LOKAL / KLIEN API ------------------------------------------------------------
    try:
        if platform == "local_lora":
            print(f"[*] Memuat Base Model & LoRA Adapter: {cfg['adapter']}...")
            tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"], trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
            base_m = AutoModelForCausalLM.from_pretrained(cfg["base_model"], quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
            from peft import PeftModel
            model = PeftModel.from_pretrained(base_m, cfg["adapter"])
            model.eval()
            
        elif platform == "local_hf":
            print(f"[*] Memuat Open-Source Model: {cfg['model_id']} (4-Bit)...")
            tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"], trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
            try:
                model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
            except Exception as e_load:
                print(f"[!] Warning remote code load ({e_load}). Mencoba load native...")
                model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], quantization_config=bnb_config, device_map="auto", trust_remote_code=False)
            model.eval()
            
        elif platform == "api_justwoker":
            if client_justwoker is None:
                print(f"[WARNING] JUSTWOKER_API_KEY tidak tersedia! Melewati {model_name}...")
                continue
            print(f"[*] Terhubung ke JustWoker Claude API: {cfg['model_id']}...")

        elif platform == "api_bluesminds":
            if client_bluesminds is None:
                print(f"[WARNING] BLUESMINDS_API_KEY tidak tersedia! Melewati {model_name}...")
                continue
            print(f"[*] Terhubung ke Bluesminds API: {cfg['model_id']}...")

        elif platform == "api_openrouter":
            if client_openrouter is None:
                print(f"[WARNING] OPENROUTER_API_KEY tidak tersedia! Melewati {model_name}...")
                continue
            print(f"[*] Terhubung ke OpenRouter API: {cfg['model_id']}...")
            
        elif platform == "api_gemini":
            if not GEMINI_API_KEY:
                print(f"[WARNING] GEMINI_API_KEY tidak tersedia! Melewati {model_name}...")
                continue
            print(f"[*] Terhubung ke Google Gemini API: {cfg['model_id']}...")
    except Exception as e_init:
        print(f"âŒ GAGAL MEMUAT MODEL {model_name}: {e_init}. Melanjutkan ke model berikutnya...")
        continue
        
    # ---- 2. INFERENSI SINGLE-TURN & MULTI-TURN ----------------------------------------------------------------
    test_suite = [("Single-Turn", simple_instances, simple_answers), ("Multi-Turn", multi_instances, multi_answers)]
    
    for set_name, instances, answers in test_suite:
        for inst in tqdm(instances, desc=f"{model_name} [{set_name}]"):
            q_id = inst["id"]
            q_msgs = inst["question"]
            gold_call_raw = answers.get(q_id, [])
            gold_calls = normalize_ground_truth(gold_call_raw)
            ref_answer_text = json.dumps(gold_calls) if gold_calls else "Tidak memerlukan tool"
            
            messages = [{"role": "system", "content": SYSTEM_PROMPT}]
            if cfg.get("mode") == "3-Shot Prompting":
                messages = list(FEW_SHOT_COT_PREFIX)
                
            if set_name == "Single-Turn":
                user_msg = q_msgs[0] if isinstance(q_msgs, list) else q_msgs
                if isinstance(user_msg, list):
                    messages.extend(user_msg)
                else:
                    messages.append(user_msg)
            else:
                for turn in q_msgs:
                    if isinstance(turn, list):
                        messages.extend(turn)
                    else:
                        messages.append(turn)
                        
            resp_text = ""
            lat = 0.0
            gen_tok = 0
            
            # EKSEKUSI BERDASARKAN PLATFORM
            if platform in ["local_lora", "local_hf"]:
                prompt_text = tokenizer.apply_chat_template(messages, tools=FORMATTED_TOOLS, tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
                t0 = time.time()
                with torch.no_grad():
                    out = model.generate(**inputs, max_new_tokens=150, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
                lat = time.time() - t0
                gen_tok = out.shape[1] - inputs.input_ids.shape[1]
                total_tokens += gen_tok
                resp_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
                
            elif platform in ["api_justwoker", "api_bluesminds", "api_openrouter"]:
                if platform == "api_justwoker":
                    client = client_justwoker
                elif platform == "api_bluesminds":
                    client = client_bluesminds
                else:
                    client = client_openrouter
                    
                t0 = time.time()
                try:
                    res = client.chat.completions.create(
                        model=cfg["model_id"],
                        messages=messages,
                        tools=FORMATTED_TOOLS,
                        temperature=0.0
                    )
                    lat = time.time() - t0
                    msg = res.choices[0].message
                    if msg.tool_calls:
                        resp_text = json.dumps([{"name": tc.function.name, "arguments": json.loads(tc.function.arguments)} for tc in msg.tool_calls])
                    else:
                        resp_text = msg.content or ""
                    gen_tok = res.usage.completion_tokens if res.usage else 30
                    total_tokens += gen_tok
                except Exception as e_api:
                    lat = time.time() - t0
                    resp_text = f"API_ERROR: {e_api}"
                    
            elif platform == "api_gemini":
                import google.generativeai as genai
                t0 = time.time()
                try:
                    g_model = genai.GenerativeModel(cfg["model_id"])
                    last_query = messages[-1]["content"] if isinstance(messages[-1], dict) else str(messages[-1])
                    res = g_model.generate_content(last_query)
                    lat = time.time() - t0
                    resp_text = res.text if hasattr(res, "text") else ""
                    gen_tok = 30
                    total_tokens += gen_tok
                except Exception as e_gem:
                    lat = time.time() - t0
                    resp_text = f"GEMINI_ERROR: {e_gem}"
                    
            pred_calls = extract_tool_calls(resp_text)
            t_acc, a_match, s_ast, f_val, err_cat = evaluate_instance(pred_calls, gold_calls)
            b4_score, rl_score = compute_nlg_metrics(resp_text, ref_answer_text)
            
            model_records.append({
                "id": q_id,
                "type": set_name,
                "gold_tool": gold_calls[0]["name"] if gold_calls else "NONE",
                "pred_tool": pred_calls[0]["name"] if pred_calls else "NONE",
                "gold_args": json.dumps(gold_calls[0]["arguments"]) if gold_calls else "{}",
                "pred_args": json.dumps(pred_calls[0]["arguments"]) if pred_calls else "{}",
                "raw_response": resp_text,
                "tool_accuracy": int(t_acc),
                "arg_match": int(a_match),
                "strict_ast": int(s_ast),
                "format_valid": int(f_val),
                "bleu4": b4_score,
                "rougeL": rl_score,
                "error_category": err_cat,
                "latency_s": lat,
                "tokens_gen": gen_tok
            })
            
    df_m = pd.DataFrame(model_records)
    total_dur = time.time() - eval_start_time
    
    clean_mname = model_name.replace("/", "_")
    summary_entry = {
        "Model Name": model_name,
        "Platform": cfg["platform"],
        "Params": cfg["params"],
        "Mode": cfg["mode"],
        "Tool Acc (%)": round(df_m["tool_accuracy"].mean() * 100, 2),
        "Macro F1 (%)": round(f1_score(df_m["gold_tool"], df_m["pred_tool"], average="macro", zero_division=0) * 100, 2),
        "Arg Match (%)": round(df_m["arg_match"].mean() * 100, 2),
        "Strict AST (%)": round(df_m["strict_ast"].mean() * 100, 2),
        "Format Valid (%)": round(df_m["format_valid"].mean() * 100, 2),
        "BLEU-4": round(df_m["bleu4"].mean(), 2),
        "ROUGE-L": round(df_m["rougeL"].mean(), 2),
        "Latency (s)": round(df_m["latency_s"].mean(), 2),
        "Throughput (tok/s)": round(total_tokens / max(0.01, total_dur), 2)
    }
    all_models_summary.append(summary_entry)
    
    json_path = os.path.join(BASE_OUTPUT_DIR, f"{clean_mname}_detailed_predictions.json")
    csv_path = os.path.join(BASE_OUTPUT_DIR, f"{clean_mname}_detailed_predictions.csv")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(model_records, f, ensure_ascii=False, indent=2)
    df_m.to_csv(csv_path, index=False)
    print(f"[OK] Log JSON tersimpan di: {json_path}")
    
    if model is not None:
        del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

print("\n[OK] SELURUH MODEL YANG AKTIF SELESAI DIEVALUASI!")


## Section 5: Tabel Komparasi 10 Pilar Multi-Platform (Format IEEE / Bab 4 Skripsi)


In [ ]:
known_baselines = [
    {"Model Name": "Base-7B-Zero-Shot", "Platform": "Baseline Fase 3", "Params": "7B", "Mode": "Zero-Shot", "Tool Acc (%)": 85.45, "Macro F1 (%)": 77.14, "Arg Match (%)": 81.82, "Strict AST (%)": 81.82, "Format Valid (%)": 98.18, "BLEU-4": 74.20, "ROUGE-L": 79.50, "Latency (s)": 5.93, "Throughput (tok/s)": 8.58},
    {"Model Name": "Base-7B-Few-Shot-CoT", "Platform": "Baseline Fase 3", "Params": "7B", "Mode": "3-Shot Prompting", "Tool Acc (%)": 87.27, "Macro F1 (%)": 81.98, "Arg Match (%)": 83.64, "Strict AST (%)": 83.64, "Format Valid (%)": 92.73, "BLEU-4": 76.80, "ROUGE-L": 81.20, "Latency (s)": 10.69, "Throughput (tok/s)": 10.12}
]

existing_names = [x["Model Name"] for x in all_models_summary]
for b in known_baselines:
    if b["Model Name"] not in existing_names:
        all_models_summary.append(b)

df_comparison = pd.DataFrame(all_models_summary)
print("="*125)
print("TABEL KOMPARASI Evaluation Metrics MULTI-PLATFORM (BAB 4 SKRIPSI / IEEE STANDARD):")
print("="*125)
print(df_comparison.to_string(index=False))
print("="*125)

SUMMARY_CSV_PATH = os.path.join(BASE_OUTPUT_DIR, "multimodel_comprehensive_benchmark_summary.csv")
df_comparison.to_csv(SUMMARY_CSV_PATH, index=False)
print(f"[OK] Tabel rekapitulasi tersimpan di: {SUMMARY_CSV_PATH}")


## Section 6: Generasi 4 Grafik Ilmiah Dinamis (IEEE/ACM Standard)


In [ ]:
print("[*] Merender 4 Grafik Ilmiah Multi-Model Dinamis...")

# 1. Multi-Model 5-Pillar Metric Comparison Bar Chart
metrics = ["Tool Acc (%)", "Macro F1 (%)", "Arg Match (%)", "Strict AST (%)", "Format Valid (%)"]
n_models = len(df_comparison)
x = np.arange(len(metrics))
width = 0.85 / max(1, n_models)

plt.figure(figsize=(18, 6), dpi=300)
cmap = plt.cm.get_cmap("tab20", n_models)

for i, row in df_comparison.iterrows():
    offset = (i - n_models / 2 + 0.5) * width
    vals = [row[m] for m in metrics]
    is_our_model = "TelkomNusa" in row["Model Name"]
    
    plt.bar(x + offset, vals, width, label=f"{row['Model Name']} ({row['Params']})",
            color=cmap(i), edgecolor="black" if is_our_model else "none",
            linewidth=1.5 if is_our_model else 0.5,
            alpha=1.0 if is_our_model else 0.8)

plt.title("Komparasi Performa Core Metrics Metrik: SLM vs Medium LLM vs MoE vs Cloud API", fontsize=13, fontweight="bold")
plt.ylabel("Persentase Skor (%)", fontsize=11)
plt.xticks(x, ["Tool Selection Acc", "Macro F1-Score", "Parameter Match Rate", "Strict Full AST", "Format Validity"], fontsize=10)
plt.ylim(30, 110)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()

CHART1_PATH = os.path.join(BASE_OUTPUT_DIR, "multimodel_5pillar_comparison.png")
plt.savefig(CHART1_PATH, bbox_inches="tight")
plt.show()
print(f"[OK] [Grafik 1] Berhasil Disimpan ke: {CHART1_PATH}")

# 2. Multi-Dimensional Radar Chart (Spider Chart Termasuk BLEU & ROUGE)
radar_metrics = ["Tool Acc (%)", "Macro F1 (%)", "Strict AST (%)", "Format Valid (%)", "BLEU-4", "ROUGE-L"]
labels = ["Tool Acc", "Macro F1", "Strict AST", "Format Valid", "BLEU-4", "ROUGE-L"]
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True), dpi=300)

for i, row in df_comparison.iterrows():
    vals = [row[m] for m in radar_metrics]
    vals += vals[:1]
    is_our_model = "TelkomNusa" in row["Model Name"]
    ax.plot(angles, vals, color=cmap(i), linewidth=2.5 if is_our_model else 1.2, label=row["Model Name"])
    ax.fill(angles, vals, color=cmap(i), alpha=0.2 if is_our_model else 0.03)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10, fontweight="bold")
ax.set_ylim(20, 105)
plt.title("Radar Chart Evaluasi Kapabilitas Multidimensi (Tool + BLEU + ROUGE)", size=12, fontweight="bold", y=1.1)
plt.legend(loc="upper right", bbox_to_anchor=(1.65, 1.1), fontsize=8)
plt.tight_layout()

CHART2_PATH = os.path.join(BASE_OUTPUT_DIR, "multimodel_radar_chart.png")
plt.savefig(CHART2_PATH, bbox_inches="tight")
plt.show()
print(f"[OK] [Grafik 2] Berhasil Disimpan ke: {CHART2_PATH}")

# 3. Efficiency Trade-Off (Latency vs Strict AST Accuracy)
plt.figure(figsize=(12, 6), dpi=300)

for i, row in df_comparison.iterrows():
    is_our_model = "TelkomNusa" in row["Model Name"]
    plt.scatter(row["Latency (s)"], row["Strict AST (%)"], color=cmap(i), s=350 if is_our_model else 180,
                edgecolor="black", linewidth=1.5 if is_our_model else 0.8, zorder=5, label=row["Model Name"])
    
    plt.annotate(f"{row['Model Name']}\n({row['Latency (s)']}s, {row['Strict AST (%)']}%)",
                 (row["Latency (s)"] + 0.15, row["Strict AST (%)"] + 0.3),
                 fontsize=8, fontweight="bold" if is_our_model else "normal")

plt.title("Trade-Off Efisiensi: Latensi vs Akurasi Strict Full AST Lintas Platform", fontsize=12, fontweight="bold")
plt.xlabel("Latensi Inferensi Rata-Rata (detik/soal) [Lebih Kecil Lebih Baik]", fontsize=11)
plt.ylabel("Strict AST Accuracy (%) [Lebih Tinggi Lebih Baik]", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

CHART3_PATH = os.path.join(BASE_OUTPUT_DIR, "multimodel_latency_vs_accuracy.png")
plt.savefig(CHART3_PATH, bbox_inches="tight")
plt.show()
print(f"[OK] [Grafik 3] Berhasil Disimpan ke: {CHART3_PATH}")


## Section 7: Ringkasan Eksekutif untuk Naskah Skripsi


In [ ]:
print("="*95)
print("RINGKASAN TEMUAN UTAMA MULTI-MODEL BENCHMARK (BAB 4 SKRIPSI):")
print("="*95)
for idx, row in df_comparison.iterrows():
    print(f"{idx+1:2d}. {row['Model Name']} [{row['Params']}]: Strict AST = {row['Strict AST (%)']}% | BLEU-4 = {row.get('BLEU-4', 0)} | ROUGE-L = {row.get('ROUGE-L', 0)} | Latency = {row['Latency (s)']}s")
print("="*95)
print("[OK] SELURUH DATA EVALUASI LINTAS MODEL & GRAFIK BERHASIL DIEKSPOR!")
